# Sparsity Attacks — Module Skills Assessment

This notebook solves the final HTB module assessment against the supplied **ResNet-18-style CIFAR-10 classifier**. It chooses ElasticNet/EAD when the server permits either method because a gradient-and-proximal attack scales much better than pairwise JSMA over `3 × 32 × 32 = 3,072` color-channel values.

The notebook is deliberately staged: inspect the challenge, reproduce the model, explain and run EAD, round-trip the candidate through PNG, verify it locally and with `/predict`, and only then optionally call `/submit_images`. Submission is **off by default**.

## 1. Imports and guarded configuration

All optimization happens in pixel space `[0,1]`. Normalization belongs inside the model wrapper because the evaluator accepts ordinary RGB PNG pixels, not normalized tensors.

In [ ]:
import base64
import hashlib
import io
import json
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

BASE_URL = 'http://154.57.164.75:31652'
REQUEST_TIMEOUT = 30
OUTPUT_DIR = Path('output/sparsity_skills_assessment')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_PATH = OUTPUT_DIR / 'cifar10_model.pth'
CANDIDATE_PATH = OUTPUT_DIR / 'ead_candidate.png'
FIGURE_PATH = OUTPUT_DIR / 'ead_comparison.png'
SUBMIT_TO_SERVER = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(1337)
print('Using device:', DEVICE)

## 2. HTTP and PNG boundary helpers

The decoded tensor has shape `(1, 3, 32, 32)`: batch, color channels, height, width. Encoding reverses that arrangement because Pillow expects height, width, channels. The formula `pixel_8bit = round(255 × pixel_01)` is read aloud as **“the eight-bit pixel equals 255 times the zero-to-one pixel, rounded.”**

In [ ]:
def http_get(path):
    with urllib.request.urlopen(f'{BASE_URL}{path}', timeout=REQUEST_TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8'))

def http_post(path, body):
    request = urllib.request.Request(
        f'{BASE_URL}{path}',
        data=json.dumps(body).encode('utf-8'),
        headers={'Content-Type': 'application/json'},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8'))

def x01_from_b64(encoded):
    image = Image.open(io.BytesIO(base64.b64decode(encoded))).convert('RGB')
    if image.size != (32, 32):
        raise ValueError(f'Expected a 32x32 image, received {image.size}.')
    hwc = np.asarray(image, dtype=np.float32) / 255.0
    return np.transpose(hwc, (2, 0, 1))[None, ...].astype(np.float32)

def b64_from_x01(x4d):
    hwc = np.transpose(np.asarray(x4d)[0], (1, 2, 0))
    pixels = np.clip(np.rint(hwc * 255.0), 0, 255).astype(np.uint8)
    buffer = io.BytesIO()
    Image.fromarray(pixels, mode='RGB').save(buffer, format='PNG', optimize=True)
    return base64.b64encode(buffer.getvalue()).decode('ascii')

health = http_get('/health')
challenge = http_get('/challenge')
metadata = http_get('/model')
items = challenge['items']
if len(items) != 1:
    raise ValueError(f'This annotated notebook expects one assessment item, received {len(items)}.')
item = items[0]
original_x01 = x01_from_b64(item['image_b64'])
SAMPLE_ID = int(item['sample_id'])
ORIGINAL_LABEL = int(item['label'])
TARGET_LABEL = int(item['target'])
REQUIRED_METHOD = str(item['required_method']).lower()
METHOD = 'ead' if REQUIRED_METHOD in {'either', 'ead'} else REQUIRED_METHOD
print({'health': health, 'sample_id': SAMPLE_ID, 'shape': original_x01.shape,
       'original_label': ORIGINAL_LABEL, 'target': TARGET_LABEL,
       'required_method': REQUIRED_METHOD, 'selected_method': METHOD})
assert METHOD == 'ead', f'This notebook implements EAD, but the server requires {REQUIRED_METHOD!r}.'

## 3. Reproduce the evaluator's ResNetCIFAR

A residual block learns a change to its input and then adds the original input back through the shortcut. In plain programming terms, it is like computing `output = learned_transform(input) + input`. That shortcut helps gradients travel through a deep network instead of repeatedly shrinking through every layer.

We verify the downloaded file's SHA-256 digest before loading it and verify that the local clean prediction matches the supplied baseline label before trusting any gradient.

In [ ]:
class BasicBlock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes),
            )
    def forward(self, x):
        transformed = F.relu(self.bn1(self.conv1(x)))
        transformed = self.bn2(self.conv2(transformed))
        return F.relu(transformed + self.shortcut(x))

class ResNetCIFAR(nn.Module):
    def __init__(self, num_blocks=(2, 2, 2, 2), num_classes=10):
        super().__init__()
        self.in_planes = 64
        self.conv1 = nn.Conv2d(3, 64, 3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.layer1 = self._make_layer(64, num_blocks[0], 1)
        self.layer2 = self._make_layer(128, num_blocks[1], 2)
        self.layer3 = self._make_layer(256, num_blocks[2], 2)
        self.layer4 = self._make_layer(512, num_blocks[3], 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, num_classes)
    def _make_layer(self, planes, count, stride):
        layers = []
        for layer_stride in [stride] + [1] * (count - 1):
            layers.append(BasicBlock(self.in_planes, planes, layer_stride))
            self.in_planes = planes
        return nn.Sequential(*layers)
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x); x = self.layer2(x); x = self.layer3(x); x = self.layer4(x)
        return self.fc(torch.flatten(self.avgpool(x), 1))

class NormalizedModel(nn.Module):
    def __init__(self, classifier, mean, std):
        super().__init__()
        self.classifier = classifier
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
    def forward(self, x01):
        return self.classifier((x01 - self.mean) / self.std)

with urllib.request.urlopen(f"{BASE_URL}{metadata['weights_url']}", timeout=REQUEST_TIMEOUT) as response:
    weights_bytes = response.read()
actual_sha256 = hashlib.sha256(weights_bytes).hexdigest()
assert actual_sha256 == metadata['weights_sha256'], 'Downloaded model hash does not match metadata.'
WEIGHTS_PATH.write_bytes(weights_bytes)
classifier = ResNetCIFAR().to(DEVICE).eval()
checkpoint = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=True)
state_dict = checkpoint.get('state_dict_ema') or checkpoint.get('state_dict') or checkpoint
classifier.load_state_dict(state_dict)
model = NormalizedModel(classifier, metadata['normalize']['mean'], metadata['normalize']['std']).to(DEVICE).eval()
original = torch.from_numpy(original_x01).to(DEVICE)
with torch.no_grad():
    clean_logits = model(original)
    local_clean_prediction = int(clean_logits.argmax(dim=1).item())
clean_server = http_post('/predict', {'image_b64': item['image_b64']})
print({'local_clean_prediction': local_clean_prediction, 'server_clean_prediction': clean_server})
assert local_clean_prediction == ORIGINAL_LABEL, 'Local model does not reproduce the supplied baseline label.'

## 4. EAD objective, pronounced and translated

We optimize

`c f(x′, t) + ||x′ − x||₂² + β ||x′ − x||₁`.

1. **Say it aloud:** “see times eff of x-prime and t, plus the squared ell-two norm of x-prime minus x, plus beta times the ell-one norm of x-prime minus x.”
2. **Pieces:** `x` is the clean image; `x′` (‘x-prime’) is the candidate; `t` is the target class; `c` weights attack success; squared L2 discourages large overall distortion; `β` (‘beta’) weights L1 sparsity.
3. **Ordinary English:** make the model choose the target, but pay for changing the image—especially for spreading changes across many values.

For a targeted attack, `f(x′,t) = max(max_{j≠t} Z_j(x′) − Z_t(x′) + κ, 0)`. Say this as: **“eff of x-prime comma t equals the maximum of: the strongest non-target logit minus the target logit plus kappa, and zero.”** `Z_j` (‘zee sub j’) is class j's raw score. When the target leads every competitor by at least `κ` (‘kappa’), this hinge loss becomes zero.

FISTA takes a gradient step for the smooth first two terms, then the L1 proximal operator soft-thresholds the perturbation `δ = x′ − x` (‘delta equals x-prime minus x’). For a scalar candidate perturbation `v`, `sign(v) max(|v| − λβ, 0)` is read **“the sign of v times the maximum of absolute v minus lambda beta, and zero.”** Values inside the threshold become exactly zero.

In [ ]:
def targeted_margin_loss(logits, target, kappa=0.0):
    target_score = logits[:, target]
    competitor_scores = logits.clone()
    competitor_scores[:, target] = -torch.inf
    strongest_competitor = competitor_scores.max(dim=1).values
    return torch.clamp(strongest_competitor - target_score + kappa, min=0.0).sum()

def soft_threshold_perturbation(candidate, original, threshold):
    delta = candidate - original
    sparse_delta = torch.sign(delta) * torch.clamp(delta.abs() - threshold, min=0.0)
    return torch.clamp(original + sparse_delta, 0.0, 1.0)

def run_fista_for_c(original, target, c_value, beta, learning_rate, kappa, iterations):
    x_previous = original.detach().clone()
    look_ahead = original.detach().clone()
    momentum_t = 1.0
    best_success = None
    best_elastic = float('inf')
    history = []
    for iteration in range(iterations):
        y = look_ahead.detach().clone().requires_grad_(True)
        logits = model(y)
        attack_loss = targeted_margin_loss(logits, target, kappa)
        l2_squared = (y - original).square().sum()
        smooth_loss = c_value * attack_loss + l2_squared
        gradient = torch.autograd.grad(smooth_loss, y)[0]
        gradient_candidate = y - learning_rate * gradient
        x_next = soft_threshold_perturbation(
            gradient_candidate, original, learning_rate * beta
        ).detach()
        next_t = (1.0 + np.sqrt(1.0 + 4.0 * momentum_t ** 2)) / 2.0
        momentum = (momentum_t - 1.0) / next_t
        look_ahead = torch.clamp(x_next + momentum * (x_next - x_previous), 0.0, 1.0)
        x_previous, momentum_t = x_next, next_t
        with torch.no_grad():
            current_logits = model(x_next)
            prediction = int(current_logits.argmax(dim=1).item())
            delta = x_next - original
            l1 = float(delta.abs().sum().item())
            l2 = float(torch.linalg.vector_norm(delta).item())
            elastic = l2 ** 2 + beta * l1
            target_probability = float(F.softmax(current_logits, dim=1)[0, target].item())
        if prediction == target and elastic < best_elastic:
            best_success, best_elastic = x_next.clone(), elastic
        if iteration % 25 == 0 or prediction == target:
            history.append({'iteration': iteration, 'prediction': prediction,
                            'target_probability': target_probability, 'l2': l2})
        if prediction == target and iteration >= 20:
            break
    return best_success, history

def elasticnet_binary_search(original, target, beta=2e-4, learning_rate=1e-2,
                             kappa=0.2, iterations=300, search_steps=6, minimum_l2=0.0):
    lower, upper, c_value = 0.0, float('inf'), 10.0
    best_overall, best_distortion, all_history = None, float('inf'), []
    for search_step in range(search_steps):
        candidate, history = run_fista_for_c(
            original, target, c_value, beta, learning_rate, kappa, iterations
        )
        all_history.append({'search_step': search_step, 'c': c_value, 'history': history})
        if candidate is not None:
            distortion = float(torch.linalg.vector_norm(candidate - original).item())
            # Normal EAD would keep the smallest successful distortion. This evaluator
            # additionally requires L2 >= minimum_l2, so only feasible successes compete.
            if distortion >= minimum_l2 and distortion < best_distortion:
                best_overall, best_distortion = candidate.clone(), distortion
            upper = min(upper, c_value)
            c_value = (lower + upper) / 2.0
        else:
            lower = max(lower, c_value)
            c_value = (lower + upper) / 2.0 if np.isfinite(upper) else c_value * 10.0
        print({'search_step': search_step + 1, 'c': all_history[-1]['c'],
               'success': candidate is not None, 'best_l2': best_distortion})
    if best_overall is None:
        raise RuntimeError('EAD did not find a target-class candidate satisfying minimum_l2.')
    return best_overall, all_history

## 5. Run EAD and satisfy the evaluator's minimum perturbation

Canonical EAD tries to minimize distortion. This particular evaluator imposes the unusual opposite safeguard `||δ||₂ ≥ 1.5`, read **“the ell-two norm of delta is greater than or equal to one point five.”** If the successful EAD result is smaller, we scale the same adversarial direction slightly outward. This is challenge-specific feasibility handling, not part of the research attack's normal objective.

In [ ]:
MINIMUM_L2 = 1.5
PNG_SAFETY_L2 = 1.55
# A larger kappa asks class 8 to lead by a robust logit margin. Besides surviving
# PNG quantization, that stronger solution naturally clears the evaluator's L2 floor.
adversarial, search_history = elasticnet_binary_search(
    original, TARGET_LABEL, kappa=5.0, minimum_l2=PNG_SAFETY_L2
)

def scale_to_minimum_l2(candidate, original, minimum_l2):
    delta = candidate - original
    current_l2 = torch.linalg.vector_norm(delta)
    if float(current_l2.item()) >= minimum_l2:
        return candidate
    scale = minimum_l2 / max(float(current_l2.item()), 1e-12)
    return torch.clamp(original + scale * delta, 0.0, 1.0)

adversarial = scale_to_minimum_l2(adversarial, original, PNG_SAFETY_L2)
with torch.no_grad():
    attack_logits = model(adversarial)
    attack_prediction = int(attack_logits.argmax(dim=1).item())
    attack_probability = float(F.softmax(attack_logits, dim=1)[0, TARGET_LABEL].item())
    attack_l2 = float(torch.linalg.vector_norm(adversarial - original).item())
print({'target': TARGET_LABEL, 'prediction': attack_prediction,
       'target_probability': attack_probability, 'float_l2': attack_l2})
assert attack_prediction == TARGET_LABEL, 'Minimum-L2 adjustment lost the targeted prediction.'
assert attack_l2 >= MINIMUM_L2, 'Candidate does not meet the evaluator minimum L2.'

## 6. PNG round-trip, measurements, and `/predict`

The server never receives our floating-point tensor; it receives an 8-bit PNG. We therefore measure and classify the decoded PNG—not merely the optimizer's in-memory result. `L0` (‘ell-zero’) counts changed channel values here, `L1` sums their absolute magnitudes, `L2` is Euclidean distance, and `L∞` (‘ell-infinity’) is the largest single change.

In [ ]:
candidate_b64 = b64_from_x01(adversarial.detach().cpu().numpy())
candidate_x01 = x01_from_b64(candidate_b64)
candidate_tensor = torch.from_numpy(candidate_x01).to(DEVICE)
decoded_delta = candidate_tensor - original
with torch.no_grad():
    decoded_logits = model(candidate_tensor)
    decoded_prediction = int(decoded_logits.argmax(dim=1).item())
metrics = {
    'l0_channel_values': int((decoded_delta.abs() > 1e-6).sum().item()),
    'l1': float(decoded_delta.abs().sum().item()),
    'l2': float(torch.linalg.vector_norm(decoded_delta).item()),
    'linf': float(decoded_delta.abs().max().item()),
}
Image.open(io.BytesIO(base64.b64decode(candidate_b64))).save(CANDIDATE_PATH)
server_prediction = http_post('/predict', {'image_b64': candidate_b64})
print({'decoded_prediction': decoded_prediction, 'target': TARGET_LABEL,
       'metrics': metrics, 'server_prediction': server_prediction,
       'candidate_path': str(CANDIDATE_PATH.resolve())})
assert decoded_prediction == TARGET_LABEL, 'PNG quantization lost the local targeted prediction.'
assert metrics['l2'] >= MINIMUM_L2, 'PNG candidate is below the required L2 threshold.'
server_class = server_prediction.get(
    'predicted_class', server_prediction.get('prediction', server_prediction.get('pred'))
)
assert int(server_class) == TARGET_LABEL, 'Server prediction does not match the target.'

In [ ]:
class_names = ('airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck')
original_hwc = np.transpose(original_x01[0], (1, 2, 0))
candidate_hwc = np.transpose(candidate_x01[0], (1, 2, 0))
magnitude = np.abs(candidate_hwc - original_hwc).max(axis=2)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(original_hwc); axes[0].set_title(f'Original: {class_names[ORIGINAL_LABEL]}')
axes[1].imshow(candidate_hwc); axes[1].set_title(f'Adversarial: {class_names[TARGET_LABEL]}')
heatmap = axes[2].imshow(magnitude, cmap='hot')
axes[2].set_title(f'Max-channel |delta|\nL2={metrics["l2"]:.3f}')
for axis in axes: axis.axis('off')
fig.colorbar(heatmap, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.savefig(FIGURE_PATH, dpi=160, bbox_inches='tight')
plt.show(block=False)  # Inline in Jupyter; non-blocking in script-based validation.
plt.close(fig)
print('Saved comparison figure:', FIGURE_PATH.resolve())

## 7. Optional final submission

The payload preserves the server's `sample_id` and uses the method signature `ead`. Leave the switch false while experimenting. Turn it on only after the preceding assertions confirm the decoded PNG reaches the target and meets the minimum L2 rule.

In [ ]:
submission_payload = {
    'items': [
        {'sample_id': SAMPLE_ID, 'method': METHOD, 'image_b64': candidate_b64}
    ]
}
if SUBMIT_TO_SERVER:
    submission_response = http_post('/submit_images', submission_payload)
    print('Submission response:', submission_response)
else:
    print('Submission skipped. Set SUBMIT_TO_SERVER=True only after reviewing all checks.')